# Querylab step 7: LoRA fine-tune (train)

Runtime: GPU (T4). Run the cells top to bottom; after a disconnect run them again, training resumes from the last checkpoint on Drive.

Before you start:
1. On your machine: `make ... build_finetune_data` writes `train.jsonl`, `validation.jsonl`, `manifest.json` (see `backend/FLOW.md`, "Fine-tuning").
2. Put those three files in Google Drive at `MyDrive/querylab/data/`.
3. Add a Colab secret `WANDB_API_KEY` (key icon in the left bar) and allow this notebook to read it. Never paste the key into a cell.

In [ ]:
# Colab ships TensorFlow, which clashes with the pinned libraries and is not used here.
# After this cell runs once, restart the runtime (Runtime > Restart session) and continue.
!pip uninstall -y -q tensorflow tensorflow-cpu tf-keras keras 2>/dev/null
!pip install -q -U "bitsandbytes>=0.48.1" peft wandb
# Colab's own transformers, torch and accelerate are kept: they are tested together.
import importlib.metadata as md
print({p: md.version(p) for p in ("torch", "transformers", "peft", "bitsandbytes", "accelerate", "wandb")})

In [ ]:
CONFIG = {
    "base_model": "Qwen/Qwen2.5-Coder-3B-Instruct",  # same as FINETUNE_BASE_MODEL
    "max_seq_len": 2048,        # longer examples are dropped and counted
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "learning_rate": 2e-4,
    "epochs": 2,
    "per_device_batch": 1,
    "grad_accum": 16,
    "warmup_ratio": 0.03,
    "seed": 42,
    "save_steps": 50,           # a checkpoint every 50 steps, so a cut-off loses little
    "run_name": "run",          # folder under drive_dir; use a new name for a different model
    "eval_steps": 100,
    "smoke_steps": 0,           # set to 50 for the first memory check, then back to 0
    "drive_dir": "/content/drive/MyDrive/querylab",
}

In [ ]:
import os
os.environ["USE_TF"] = "0"  # transformers: never import TensorFlow
os.environ["USE_FLAX"] = "0"
import hashlib, json, glob
from google.colab import drive, userdata
drive.mount("/content/drive")
DATA = f"{CONFIG['drive_dir']}/data"
RUN = f"{CONFIG['drive_dir']}/{CONFIG['run_name']}"
os.makedirs(RUN, exist_ok=True)

manifest = json.load(open(f"{DATA}/manifest.json"))
for name, digest in manifest["sha256"].items():
    actual = hashlib.sha256(open(f"{DATA}/{name}", "rb").read()).hexdigest()
    assert actual == digest, f"{name} does not match manifest.json: rebuild or re-upload"
print(manifest["train_examples"], "train,", manifest["validation_examples"], "validation")

os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ["WANDB_PROJECT"] = "querylab-lora"

In [ ]:
import torch
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"])
tok.padding_side = "right"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def encode(record):
    # Loss only on the assistant reply: the prompt tokens are masked with -100.
    system, user, assistant = record["messages"]
    prompt_text = tok.apply_chat_template([system, user], add_generation_prompt=True, tokenize=False)
    prompt_ids = tok(prompt_text, add_special_tokens=False)["input_ids"]
    reply_ids = tok(assistant["content"] + tok.eos_token, add_special_tokens=False)["input_ids"]
    ids = prompt_ids + reply_ids
    if len(ids) > CONFIG["max_seq_len"]:
        return None
    return {"input_ids": ids, "attention_mask": [1] * len(ids),
            "labels": [-100] * len(prompt_ids) + reply_ids}

def load(name):
    rows = [json.loads(line) for line in open(f"{DATA}/{name}")]
    encoded = [encode(r) for r in rows]
    kept = [e for e in encoded if e]
    print(name, len(kept), "kept,", len(rows) - len(kept), "dropped as too long")
    return kept

train_rows, val_rows = load("train.jsonl"), load("validation.jsonl")

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], quantization_config=bnb,
                                             device_map={"": 0}, torch_dtype=torch.float16)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(
    r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=CONFIG["lora_dropout"],
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM"))
model.print_trainable_parameters()

In [ ]:
import wandb
from transformers import Trainer, TrainingArguments

def collate(batch):
    longest = max(len(b["input_ids"]) for b in batch)
    pad = tok.pad_token_id
    return {
        "input_ids": torch.tensor([b["input_ids"] + [pad] * (longest - len(b["input_ids"])) for b in batch]),
        "attention_mask": torch.tensor([b["attention_mask"] + [0] * (longest - len(b["attention_mask"])) for b in batch]),
        "labels": torch.tensor([b["labels"] + [-100] * (longest - len(b["labels"])) for b in batch]),
    }

# Whole steps, not a ratio: newer transformers versions removed `warmup_ratio`.
steps_per_epoch = -(-len(train_rows) // (CONFIG["per_device_batch"] * CONFIG["grad_accum"]))
warmup_steps = max(1, round(CONFIG["warmup_ratio"] * steps_per_epoch * CONFIG["epochs"]))
print("steps per epoch:", steps_per_epoch, "warmup steps:", warmup_steps)

args = TrainingArguments(
    output_dir=f"{RUN}/checkpoints",
    num_train_epochs=CONFIG["epochs"],
    max_steps=CONFIG["smoke_steps"] or -1,
    per_device_train_batch_size=CONFIG["per_device_batch"],
    per_device_eval_batch_size=CONFIG["per_device_batch"],
    gradient_accumulation_steps=CONFIG["grad_accum"],
    learning_rate=CONFIG["learning_rate"],
    warmup_steps=warmup_steps,
    lr_scheduler_type="cosine",
    fp16=True,
    gradient_checkpointing=True,
    logging_steps=10,
    eval_strategy="steps", eval_steps=CONFIG["eval_steps"],
    save_strategy="steps", save_steps=CONFIG["save_steps"], save_total_limit=3,
    seed=CONFIG["seed"], data_seed=CONFIG["seed"],
    report_to="wandb",
    run_name=f"lora-{CONFIG['base_model'].split('/')[-1]}-{manifest['sha256']['train.jsonl'][:8]}",
    remove_unused_columns=False,
)
trainer = Trainer(model=model, args=args, train_dataset=train_rows, eval_dataset=val_rows, data_collator=collate)
# Only checkpoints that finished writing (trainer_state.json is written last).
found = [p for p in glob.glob(f"{RUN}/checkpoints/checkpoint-*") if os.path.exists(f"{p}/trainer_state.json")]
last = max(found, key=lambda p: int(p.rsplit("-", 1)[1]), default=None)
print("resuming from", last)
trainer.train(resume_from_checkpoint=last)
print("peak GPU memory (GB):", torch.cuda.max_memory_allocated() / 1e9)

In [ ]:
if CONFIG["smoke_steps"]:
    print("Smoke run only: set smoke_steps to 0 and run again for the real training.")
else:
    trainer.save_model(f"{RUN}/adapter")
    tok.save_pretrained(f"{RUN}/adapter")
    digest = hashlib.sha256(open(f"{RUN}/adapter/adapter_model.safetensors", "rb").read()).hexdigest()
    record = {**CONFIG, "adapter_sha256": digest, "data_manifest": manifest,
              "wandb_run": wandb.run.url if wandb.run else None,
              "final_eval_loss": trainer.evaluate()["eval_loss"]}
    json.dump(record, open(f"{RUN}/adapter/train_config.json", "w"), indent=2)
    print(json.dumps({k: record[k] for k in ("adapter_sha256", "final_eval_loss", "wandb_run")}, indent=2))